# Comparing ERA5 with observed datasets

ERA5 is a reanalysis: a weather model that takes in observations. This notebook compares ERA5 with three gridded datasets made from observations:

- [IMD gridded rainfall](https://www.imdpune.gov.in/cmpg/Griddata/Rainfall_25_NetCDF.html) (0.25°, daily), for monthly rainfall in each Indian state.
- [IMD gridded temperature](https://www.imdpune.gov.in/cmpg/Griddata/Max_1_Bin.html) (1°, daily), for daily maximum and minimum temperature in Maharashtra.
- [CRU TS v4.07](https://crudata.uea.ac.uk/cru/data/hrg/) (0.5°, monthly), for monthly temperature over India.

Each section puts the two datasets on one grid or one set of regions. Then `compare` calculates the number of values `n`, the Pearson correlation `r`, the bias (ERA5 - observed) and the RMSE. [ERA5](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels) needs a CDS account. IMD and CRU TS download without credentials.

In [ ]:
import pandas as pd

import varunayan as v

states = v.load_india_boundaries("state")

## Monthly rainfall in each state: ERA5 and IMD (2000-2024)

`era5ify_geojson` with `level="feature"` gives one row for each state and month, with the mean of the ERA5 cells inside the state. `imd_rainfall_bbox` adds the daily IMD values of each month. Both values are monthly totals in mm. The IMD column `period` holds the month.

In [ ]:
era5_rain = v.era5ify_geojson(
    request_id="era5_india_states_rain",
    variables=["total_precipitation"],
    start_date="2000-01-01",
    end_date="2024-12-31",
    json_file="https://geo.bharatviz.org/geojsons/admin/India-geodata-lgd-states.geojson",
    frequency="monthly",
    resolution=0.25,
    level="feature",
)
imd_rain = v.imd_rainfall_bbox(
    request_id="imd_india_rain",
    start_year=2000,
    end_year=2024,
    north=38.5,
    south=6.5,
    east=100.0,
    west=66.5,
    frequency="monthly",
)
era5_rain.head()

`aggregate_to_polygons` takes the mean of the IMD cells inside each state for each month. With `fill_empty=True`, a state smaller than a grid cell gets the nearest cell, as in the ERA5 step.

In [ ]:
imd_states = v.aggregate_to_polygons(
    imd_rain,
    states,
    "rainfall",
    polygon_id_col="state_name",
    by=["year", "period"],
    fill_empty=True,
)
rain = pd.merge(
    era5_rain.rename(columns={"value": "era5"}),
    imd_states.rename(columns={"period": "month", "rainfall": "imd"}),
    on=["state_name", "year", "month"],
)
rain_stats = v.compare(rain, "imd", "era5", by="state_name")
rain_stats.sort_values("r")

In [ ]:
v.plot_bars(
    rain_stats,
    "state_name",
    "r",
    horizontal=True,
    sort=True,
    title="Correlation of monthly rainfall, IMD vs ERA5",
)

In [ ]:
v.plot_map(
    states.merge(rain_stats, on="state_name"),
    "bias",
    cmap="RdBu_r",
    center=0,
    title="Mean bias of monthly rainfall, 2000-2024",
    legend_title="ERA5 - IMD (mm)",
)

In 31 of the 36 states, r is 0.85 or more. The agreement is best in Madhya Pradesh, Rajasthan and Maharashtra (r = 0.99). The agreement is worst in Andaman and Nicobar and in Ladakh (r = 0.59). These regions are islands and high mountains, with few IMD stations.

In most states, ERA5 gives more rain than IMD (median bias +7.5 mm in each month). The largest positive bias is in Arunachal Pradesh (+142 mm) and Sikkim (+106 mm). The largest negative bias is in Dadra and Nagar Haveli and Daman and Diu (-88 mm) and in Goa (-77 mm).

## Daily temperature extremes in Maharashtra: ERA5 and IMD (2020)

For daily data, `era5ify_bbox` takes the maximum of `mx2t` and the minimum of `mn2t` in each day (°C). We request ERA5 at 0.25° in a box whose edges are on the edges of the IMD 1° cells. Then `regrid` takes the mean of the 0.25° cells in each 1° cell. The 1° cell centers are at x.5°, the same as the IMD grid.

In [ ]:
era5_temp = v.era5ify_bbox(
    request_id="era5_maharashtra_tmax_tmin",
    variables=[
        "maximum_2m_temperature_since_previous_post_processing",
        "minimum_2m_temperature_since_previous_post_processing",
    ],
    start_date="2020-01-01",
    end_date="2020-12-31",
    north=21.75,
    south=15.0,
    east=80.75,
    west=72.0,
    frequency="daily",
    resolution=0.25,
)
era5_temp = v.regrid(era5_temp, "value", 1.0, by=["datetime", "variable"]).rename(
    columns={"datetime": "date", "value": "era5"}
)
era5_temp.head()

`imd_temperature_bbox` gives the IMD daily `tmax` or `tmin` (°C). We put ERA5 `mx2t` next to IMD `tmax`, and ERA5 `mn2t` next to IMD `tmin`.

In [ ]:
box = dict(north=21.5, south=15.5, east=80.5, west=72.5)
keys = ["date", "latitude", "longitude"]
tmax = pd.merge(
    era5_temp[era5_temp["variable"] == "mx2t"],
    v.imd_temperature_bbox("imd_mh_tmax", 2020, 2020, var_type="tmax", **box),
    on=keys,
).rename(columns={"temperature": "imd"})
tmin = pd.merge(
    era5_temp[era5_temp["variable"] == "mn2t"],
    v.imd_temperature_bbox("imd_mh_tmin", 2020, 2020, var_type="tmin", **box),
    on=keys,
).rename(columns={"temperature": "imd"})
grid = ["latitude", "longitude"]
temp_stats = pd.merge(
    v.compare(tmax, "imd", "era5", by=grid),
    v.compare(tmin, "imd", "era5", by=grid),
    on=grid,
    suffixes=("_tmax", "_tmin"),
)
temp_stats.head()

In [ ]:
maharashtra = states[states["state_name"] == "MAHARASHTRA"]
v.plot_map(
    temp_stats,
    ["r_tmax", "r_tmin"],
    boundaries=maharashtra,
    title="Correlation of daily temperature, IMD vs ERA5 (2020)",
    legend_title="Pearson r",
)

In [ ]:
v.plot_map(
    temp_stats,
    ["bias_tmax", "bias_tmin"],
    boundaries=maharashtra,
    cmap="RdBu_r",
    center=0,
    title="Mean bias of daily temperature (2020)",
    legend_title="ERA5 - IMD (°C)",
)

For both variables, the median r over the cells is 0.97. ERA5 tmax is colder than IMD in all cells (median bias -1.9 °C). ERA5 tmin is warmer than IMD in most cells (median bias +1.0 °C). Thus, the daily temperature range of ERA5 is smaller than the range of IMD. The agreement is worst in the coastal cells at 72.5° E near Mumbai (r = 0.51 for tmax). These 1° cells contain sea.

## Monthly temperature over India: ERA5 and CRU TS (2000-2018)

`cru_ts_bbox` downloads CRU TS directly from UEA. `regrid` averages the 0.25° ERA5 cells onto the 0.5° CRU TS grid. CRU TS has data only on land, so the merge also removes the ERA5 ocean cells.

In [ ]:
india = dict(north=37, south=6, east=98, west=68)
era5_t2m = v.era5ify_bbox(
    request_id="india_era5_t2m_monthly",
    variables=["2m_temperature"],
    start_date="2000-01-01",
    end_date="2018-12-31",
    frequency="monthly",
    **india,
)
cru = v.cru_ts_bbox(variable="tmp", start_year=2000, end_year=2018, **india)
cru_both = pd.merge(
    v.regrid(era5_t2m, "value", 0.5, by=["year", "month"]),
    cru.rename(columns={"value": "cru"}),
    on=["year", "month", "latitude", "longitude"],
).rename(columns={"value": "era5"})
cru_both.head()

`spatial_mean` gives the area-weighted mean over India for each month. The box includes the Himalaya and part of the Tibetan Plateau, so the means are lower than the means for India alone.

In [ ]:
monthly = pd.merge(v.spatial_mean(cru_both, "era5"), v.spatial_mean(cru_both, "cru"))
v.compare(monthly, "cru", "era5")

In [ ]:
v.plot_series(
    monthly.melt(id_vars="date", value_vars=["era5", "cru"], var_name="source"),
    x="date",
    y="value",
    by="source",
    ylabel="Mean temperature (°C)",
    title="India: monthly mean temperature, ERA5 and CRU TS",
)

`seasonal_stat` with all 12 months and `how="mean"` gives the 2000-2018 mean in each cell. The map shows the difference ERA5 - CRU TS.

In [ ]:
cells = v.seasonal_stat(
    cru_both.assign(bias=cru_both["era5"] - cru_both["cru"]),
    "bias",
    months=range(1, 13),
    how="mean",
)
v.plot_map(
    cells,
    "bias",
    boundaries=states,
    cmap="RdBu_r",
    center=0,
    title="Mean temperature bias, 2000-2018",
    legend_title="ERA5 - CRU TS (°C)",
)

The two monthly series follow each other closely (r = 0.999), and ERA5 is approximately 0.9 °C colder than CRU TS. Over most of the plains, the difference is small. The largest differences are in the mountains, where the two datasets use different surface heights.

## When to use which dataset

| Feature | ERA5 | IMD gridded | CRU TS v4.07 |
|---------|------|-------------|--------------|
| Resolution | 0.25° (or coarser) | 0.25° rain, 1° temperature | 0.5° |
| Coverage | Global, 1940 to present | India, rain from 1901, temperature from 1951 | Global land, 1901-2022 |
| Frequency | Hourly, daily, monthly | Daily | Monthly |
| Source | Reanalysis (model and observations, land and ocean) | Interpolated IMD stations (land) | Interpolated stations (land) |
| Credentials | Yes (CDS account) | No | No |
| Variables | 100+, including wind, radiation, soil moisture | Rainfall, tmax, tmin | tmp, tmx, tmn, pre, vap, cld, wet, frs, dtr, pet |

Use IMD for observed rainfall and temperature in India. Use CRU TS for long historical land baselines without CDS credentials. Use ERA5 for sub-daily data, ocean coverage, regions outside India, or variables other than temperature and rainfall.